# Unit 7 · Classification metrics

How good is the spam filter from Unit 6? Start with the same Spambase measurements, recorded labels, and fitted coefficients. Then choose a cutoff on validation and evaluate it on the original held-out emails.

[Open the interactive lesson](https://sharanharsoor.github.io/ml-learning-lab/ml-foundations/#unit-7) · [UCI Spambase](https://doi.org/10.24432/C53G6X) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

In [ ]:
import json
import os
from math import exp
from pathlib import Path
from urllib.request import urlopen

url = (
    "https://raw.githubusercontent.com/sharanharsoor/ml-learning-lab/"
    "main/labs/ml-foundations/data/unit-6.json"
)
if local := os.getenv("ML_FOUNDATIONS_SPAM_FIXTURE"):
    fixture = json.loads(Path(local).read_text())
else:
    with urlopen(url, timeout=20) as response:
        fixture = json.load(response)

b, w = fixture["fit"]["intercept"], fixture["fit"]["weight"]

def probability(x):
    z = b + w * x
    if z >= 0:
        return 1 / (1 + exp(-z))
    odds = exp(z)
    return odds / (1 + odds)

def scored(split):
    return [{**row, "p": probability(row["x"])} for row in fixture["rows"][split]]

validation = scored("validation")
test = scored("test")
print(f"{len(validation):,} validation emails and {len(test):,} held-out emails")

## Place four emails in the matrix

Rows are recorded labels. Columns are predicted labels. At cutoff 0.50, follow E17, E3027, E1365, and E3383 through all four cells. Change the cutoff to 0.35 and check which decisions change.

In [ ]:
cutoff = 0.50
labels = {0: "Legitimate", 1: "Spam"}
cells = {(1, True): "TP", (0, True): "FP", (1, False): "FN", (0, False): "TN"}

def outcome(row, cutoff):
    return cells[row["y"], row["p"] >= cutoff]

by_id = {row["id"]: row for row in validation}
for email in ["E17", "E3027", "E1365", "E3383"]:
    row = by_id[email]
    print(email, f'{row["p"]:.4f}', labels[row["y"]],
          "→", labels[row["p"] >= cutoff], outcome(row, cutoff))

## Read each denominator

Precision uses **all Spam predictions**. Recall uses **all recorded Spam**. False-positive rate uses **all recorded Legitimate**. Accuracy counts correct predictions across all rows. F1 combines precision and recall. Use the count formula to avoid rounding intermediate rates.

In [ ]:
def matrix(rows, cutoff):
    result = dict.fromkeys(["TP", "FP", "FN", "TN"], 0)
    for row in rows:
        result[outcome(row, cutoff)] += 1
    return result

def report(result):
    tp, fp, fn, tn = (result[cell] for cell in ["TP", "FP", "FN", "TN"])
    values = {
        "Accuracy": (tp + tn, tp + fp + fn + tn),
        "Precision": (tp, tp + fp),
        "Recall": (tp, tp + fn),
        "False-positive rate": (fp, fp + tn),
        "F1": (2 * tp, 2 * tp + fp + fn),
    }
    print(result)
    for name, (numerator, denominator) in values.items():
        fraction = f"{numerator} ÷ {denominator} = {numerator / denominator:.2%}" if denominator else "Undefined"
        print(f"{name:<21} {fraction}")

report(matrix(validation, 0.50))

## Choose a policy on validation

Choose among 35%, 50%, and 65%. Catch as much spam as possible under a limit of 25 legitimate emails flagged on validation. Freeze that choice before scoring test.

In [ ]:
options = []
for cutoff in fixture["thresholds"]["presets"]:
    result = matrix(validation, cutoff)
    options.append((cutoff, result))
    print(f"CUTOFF {cutoff:.2f} | caught {result['TP']} | false alarms {result['FP']}")

eligible = [(cutoff, result) for cutoff, result in options if result["FP"] <= 25]
chosen, result = max(eligible, key=lambda option: option[1]["TP"])
print(f"Locked at {chosen:.2f} using validation.")

In [ ]:
print("Original held-out test split")
report(matrix(test, chosen))
print("Always inbox on the same test emails")
report(matrix(test, float("inf")))

The held-out fold flags **27** legitimate emails at the chosen cutoff. The validation limit helped select the rule. The separate fold shows how the fixed rule performs on a different sample.

## Sweep the same validation scores

Each exact score group enters together. ROC puts false-positive rate on the horizontal axis and recall on the vertical axis. For AUC, each Spam–Legitimate pair earns one point if Spam scores higher, half if tied, and zero if lower.

In [ ]:
from itertools import groupby
import matplotlib.pyplot as plt

positive_count = sum(row["y"] for row in validation)
negative_count = len(validation) - positive_count
points = [(0.0, 0.0)]
tp = fp = wins = ties = 0
by_score = sorted(validation, key=lambda row: row["p"], reverse=True)
for score, group in groupby(by_score, key=lambda row: row["p"]):
    members = list(group)
    new_spam = sum(row["y"] for row in members)
    new_legitimate = len(members) - new_spam
    tp += new_spam
    fp += new_legitimate
    wins += new_spam * (negative_count - fp)
    ties += new_spam * new_legitimate
    points.append((fp / negative_count, tp / positive_count))

pairs = positive_count * negative_count
auc = (wins + 0.5 * ties) / pairs
print(f"Validation AUC = ({wins:,} + 0.5 × {ties:,}) ÷ {pairs:,} = {auc:.4f}")
print("Exact tied pair", by_id["E1405"]["p"], by_id["E3487"]["p"])
current = matrix(validation, 0.50)
selected = (current["FP"] / negative_count, current["TP"] / positive_count)
fig, ax = plt.subplots(figsize=(5.7, 5))
x, y = zip(*points)
ax.plot(x, y, label=f"Validation ROC · AUC {auc:.4f}", color="#0a6b5c")
ax.plot([0, 1], [0, 1], "--", color="#83958a", label="Chance ranking")
ax.scatter(*selected, color="#624789", zorder=3, label="0.50 cutoff")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="False-positive rate", ylabel="Recall (true-positive rate)")
ax.legend(loc="lower right")
plt.show()

## Continue in the Classification Metrics lab

Open **Read matrix and rates**, then **Read ROC and AUC**. That lesson uses the same source dataset with all 57 measurements and a different split, so its worked scores and counts belong to that model.

[Classification Metrics lab](https://sharanharsoor.github.io/ml-learning-lab/classification-regression-metrics/#classification)